# Notebook 09 — Re-ingestion + staleness diff

Slice 13 makes CFR ingestion re-runnable safely. Every CFR-derived node carries `retrieved_at` (ISO date) and `content_hash` (sha256). `diff_section` compares the live eCFR XML against the graph state and reports per-DC drift. `apply_diff` is the only function that mutates the graph based on a diff — the two are intentionally decoupled.

**Prereqs**

- `make up` — Neo4j running
- §4.71a already ingested (run notebook 02 first)
- `OPENAI_API_KEY` set (only needed if a DC actually changed)

## 1. Inspect existing graph state

Peek at a couple of DCs in §4.71a to see the `content_hash` + `retrieved_at` columns the writer set during ingestion.

In [ ]:
from va_agent.graph.driver import GraphDriver

with GraphDriver.from_env() as driver:
    rows = driver.cfr_read(
        """
        MATCH (dc:CFR:DiagnosticCode)-[:IN_SECTION]->(s:CFR:Section {id: '4.71a'})
        RETURN dc.code AS code, dc.title AS title,
               substring(dc.content_hash, 0, 12) AS hash_prefix,
               dc.retrieved_at AS retrieved_at
        ORDER BY dc.code
        LIMIT 5
        """
    )
    for row in rows:
        print(row)

## 2. Diff §4.71a against the live eCFR

`diff_section` fetches the section XML (cached), discovers every DC in the fresh XML, and for each existing DC compares its freshly-extracted `content_hash` against what's in the graph. Because we ingested §4.71a from the same `retrieval_date`, this should report **all unchanged**.

In [ ]:
from datetime import date
from va_agent.ingestion.diff import diff_section

with GraphDriver.from_env() as driver:
    report = diff_section('4.71a', driver, retrieval_date=date(2025, 1, 1))

print(f'unchanged: {len(report.unchanged)}')
print(f'changed:   {len(report.changed)}')
print(f'removed:   {len(report.removed)}')
print(f'new:       {len(report.new)}')
print(f'failed:    {len(report.failed)}')
print(f'has_changes: {report.has_changes}')

## 3. Manually mutate one Criterion to simulate drift

To exercise the diff machinery against a known change, hand-corrupt the stored `content_hash` of DC 5260 so it no longer matches the freshly-extracted hash.

In [ ]:
with GraphDriver.from_env() as driver:
    driver.cfr_write(
        "MATCH (dc:CFR:DiagnosticCode {code: '5260'}) SET dc.content_hash = 'stale-hash'"
    )
    # Also mutate one criterion's text so we can visibly verify apply_diff restores it.
    driver.cfr_write(
        """
        MATCH (dc:CFR:DiagnosticCode {code: '5260'})-[:HAS_RATING]->(:CFR:RatingLevel)
              -[:REQUIRES]->(c:CFR:Criterion)
        WITH c LIMIT 1
        SET c.text = 'CORRUPTED TEXT'
        """
    )
print('mutated DC 5260')

## 4. Re-run diff — DC 5260 now shows up as `changed`

In [ ]:
with GraphDriver.from_env() as driver:
    report = diff_section('4.71a', driver, retrieval_date=date(2025, 1, 1))

print(f'changed DCs: {[c.code for c in report.changed]}')
for c in report.changed:
    print(f'  {c.code}: {c.old_hash[:12]}... -> {c.new_hash[:12]}...')

## 5. Apply the diff to restore canonical state

`apply_diff(report, confirm=True)` deletes the changed-DC subgraph (DC + Criteria + Measurements + Notes + CrossReferences) and re-writes it from the fresh extraction. `confirm=False` raises — the flag exists so a caller can't apply by accident.

In [ ]:
from va_agent.ingestion.diff import apply_diff

with GraphDriver.from_env() as driver:
    apply_report = apply_diff(report, driver, confirm=True)

print('rewrote:', apply_report.rewrote)
print('deleted:', apply_report.deleted)
print('added:  ', apply_report.added)
print('skipped:', apply_report.skipped)

## 6. Verify the diff is empty again

In [ ]:
with GraphDriver.from_env() as driver:
    report = diff_section('4.71a', driver, retrieval_date=date(2025, 1, 1))

print(f'has_changes: {report.has_changes}')
print(f'unchanged: {len(report.unchanged)}, changed: {len(report.changed)}, '
      f'removed: {len(report.removed)}, new: {len(report.new)}')

# And confirm the corrupted criterion text is gone.
with GraphDriver.from_env() as driver:
    orphans = driver.cfr_read(
        "MATCH (c:CFR:Criterion {text: 'CORRUPTED TEXT'}) RETURN count(c) AS n"
    )
print('orphan corrupted criteria:', orphans[0]['n'])